In [1]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
import requests
import pickle
import random
from scipy.interpolate import griddata
# from scipy.optimize import fsolve
# from scipy.optimize import brentq
from scipy.optimize import minimize_scalar
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
# pip install --upgrade pypa

In [2]:
from age_pension import calculate_pension_income
from CHSP_GRandF import GRandF_CHSP
from HCP_GRandF import GRandF_HCP
from RC_GRandF import GRandF_RC
from comInterp import compress_interp
from FIstatus import I_status, F_status
from DeltaW import Delta_W_status
from updown_prep import derivatives_tp1, direct2EI, Hstate2Cf, reVC, Compare35
from AIP import xiatt, Hstate2IRACF

In [3]:
def return_calib_var(alpha = 0.5):
    Leasing = 1
    W = 1200000
    H = W * alpha/(alpha+1)
    tau1_arr = np.full(41, 1/6)
    tau2_HCP4 = np.full(41, 3/4)
    tau2_RC = np.full(41, 1/24)
    return H, W/(1+alpha), tau1_arr, tau2_HCP4, tau2_RC, Leasing

In [4]:
ratios = np.linspace(0.05, 0.95, 25)  # Create 41 points from 0 to 1 exclusive
alphas = ratios / (1 - ratios)  # Solve alpha / (1 + alpha) = ratio for alpha

In [5]:
C_low = 10000
C_RACF = 17961
rho = 2
gamma = 5
theta = (1-gamma)/(1-rho)
b = 2
beta = 0.96
# W_max = 2000000
# _,Wd1pa,_,_,_,_= return_calib_var(0)
# lambda1 = Wd1pa/W_max 

In [6]:
with open('National_Transit.pkl', 'rb') as file:
    National_Transit = pickle.load(file)

with open('ME_grid_H.pkl', 'rb') as file:
    ME_grid_H = pickle.load(file)

with open('ME_grid_ir.pkl', 'rb') as file:
    ME_grid_ir = pickle.load(file)

with open('ME_grid_cpi.pkl', 'rb') as file:
    ME_grid_cpi = pickle.load(file)

ME_grid_cpi[0] = [1]
ME_grid_H[0] = [1]

In [7]:
def wealth_law_35(EI_t, W_t, C_t, alpha, t, Hadded, Hstate):
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    r_t = ME_grid_ir[t][EI_t]/100
    cpi_t = ME_grid_cpi[t][EI_t]
    H_t = H_ini * ME_grid_H[t][EI_t] / cpi_t
    I_t = I_status(W_t, H_t, Hstate)
    F_t = F_status(W_t, H_t, Hstate)
    W_tp1 = (W_t - C_t + I_t - F_t)*(1 + r_t) + H_t*Hadded
    return W_tp1

In [8]:
def up_down_situation_35(EI_tm1, W_t, C_t, alpha, Hstate, t, direction, Dict_interp_tp1):
    EI_t = direct2EI(EI_tm1, direction)
    C_f_t = Hstate2Cf(Hstate)
    W_tp1 = max(wealth_law_35(EI_t, W_t, C_t, alpha, t, 0, Hstate), C_f_t)
    W_tp1_selling = max(wealth_law_35(EI_t, W_t, C_t, alpha, t, 1, Hstate), 1)
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    r_t = ME_grid_ir[t][EI_t]/100
    cpi_t = ME_grid_cpi[t][EI_t]
    H_t = H_ini * ME_grid_H[t][EI_t] / cpi_t
    I_RACF = Hstate2IRACF(Hstate)
    if t == 40:
        expectation_1 = b ** gamma * W_tp1_selling ** (1 - gamma)
        expectation_2 = (1 + r_t) * b ** gamma / (1 - beta)* (1/xiatt(t,alpha,I_RACF))**(1-rho) * W_tp1_selling ** (-gamma)
    if t < 40:
        pi_t = National_Transit[t]
        V3_interpolated = Dict_interp_tp1['V'][Hstate][EI_t]
        C3_interpolated = Dict_interp_tp1['C'][Hstate][EI_t]
        V3 = V3_interpolated(W_tp1).item()
        C3 = C3_interpolated(W_tp1).item()
        mu3s = 1 + derivatives_tp1(W_tp1, H_t, Hstate)
        expectation_1 = pi_t[2, 2]*V3**(1-gamma) + pi_t[2, 3]*b**gamma*W_tp1_selling**(1-gamma)
        expectation_2 = pi_t[2, 2]*(1 + r_t)*V3**(rho-gamma)*C3**(-rho)*mu3s*(xiatt(t+1,alpha,I_RACF)/xiatt(t,alpha,I_RACF))**(1-rho)\
                        + pi_t[2, 3]*(1 + r_t)*b**gamma/(1-beta)*(1/xiatt(t,alpha,I_RACF))**(1-rho)*W_tp1_selling**(-gamma)
    return expectation_1, expectation_2

In [9]:
def C_selection_t_35(EI_tm1, W_t, C_t, alpha, Hstate, t, Dict_interp_tp1):
    expectation_1_up, expectation_2_up = up_down_situation_35(EI_tm1, W_t, C_t, alpha, Hstate, t,'up', Dict_interp_tp1) # goes up 
    expectation_1_down, expectation_2_down = up_down_situation_35(EI_tm1, W_t, C_t, alpha, Hstate, t,'down', Dict_interp_tp1) #  goes down 
    expectation_1 = (expectation_1_up + expectation_1_down)/2
    expectation_2 = (expectation_2_up + expectation_2_down)/2
    C_t_star = (beta*(expectation_1**(1/theta - 1))*expectation_2) ** (-1/rho)
    return (C_t_star - C_t)

In [10]:
def find_C_t_35(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
    C_f_t = C_RACF if Hstate == 5 else C_low
    def function_to_minimize(C_t, EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
    # Define your equation here
        return abs(C_selection_t_35(EI_tm1, W_t, C_t, alpha, Hstate, t, Dict_interp_tp1))
    result = minimize_scalar(function_to_minimize, args=(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1), bounds=(C_f_t, W_t), method='bounded')
    return result.x


In [11]:
def V_t35(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
    I_RACF = Hstate2IRACF(Hstate)
    optimal_C_t = find_C_t_35(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1)
    expectation_1_up, expectation_2_up     = up_down_situation_35(EI_tm1, W_t, optimal_C_t, alpha, Hstate, t, 'up',   Dict_interp_tp1) # goes up 
    expectation_1_down, expectation_2_down = up_down_situation_35(EI_tm1, W_t, optimal_C_t, alpha, Hstate, t, 'down', Dict_interp_tp1) #  goes down 
    expectation_1 = (expectation_1_up + expectation_1_down)/2
    V = ((1-beta) * (xiatt(t,alpha,I_RACF) * optimal_C_t)**(1-rho) + beta*expectation_1**(1/theta))**(1/(1-rho))
    return V, optimal_C_t


In [12]:
# Generate the grids for downsizing options, varying on the intial relocation of liquid and illiquid wealth 
def gen_grid_35(alpha, numpoints, Hstate):
    Dict_grid = {}
    C_upper = 60000
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    C_f_t = C_RACF if Hstate == 5 else C_low
    W_lowest = C_f_t + 10
    W_lower_previous = W_lowest
    W_upper_previous = W_ini
    for t in range(1, 41):
        grid = np.linspace(W_lower_previous, W_upper_previous, numpoints)
        Dict_grid[t]= grid
        W_upper = wealth_law_35(int(t/2), W_upper_previous, C_RACF,   alpha, t, 0, 1)
        W_upper_previous = W_upper
        W_lower = wealth_law_35(int(t/2), W_lower_previous, C_upper, alpha, t, 0, 5)
        W_lower_previous = max(W_lower, W_lowest)
    return Dict_grid


In [13]:
def add_interpolated_CV_35(t, alpha, Dict_grids, V3_t, V5_t):
    Dict_interpolated_V3_= {}
    Dict_interpolated_V5_= {}
    Dict_interpolated_C3_= {}
    Dict_interpolated_C5_= {}
    for EI_t in range(t):
        V3_array = np.array([V3_t[(EI_t, W)][0] for W in Dict_grids[3][t]])
        C3_array = np.array([V3_t[(EI_t, W)][1] for W in Dict_grids[3][t]])
        V3_interpolated = interp1d(Dict_grids[3][t], V3_array, kind='linear', fill_value= "extrapolate")
        C3_interpolated = interp1d(Dict_grids[3][t], C3_array, kind='linear', fill_value= "extrapolate")
        V5_array = np.array([V5_t[(EI_t, W)][0] for W in Dict_grids[5][t]])
        C5_array = np.array([V5_t[(EI_t, W)][1] for W in Dict_grids[5][t]])
        V5_interpolated = interp1d(Dict_grids[5][t], V5_array, kind='linear', fill_value= "extrapolate")
        C5_interpolated = interp1d(Dict_grids[5][t], C5_array, kind='linear', fill_value= "extrapolate")
        Dict_interpolated_V3_[EI_t] = V3_interpolated
        Dict_interpolated_V5_[EI_t] = V5_interpolated
        Dict_interpolated_C3_[EI_t] = C3_interpolated
        Dict_interpolated_C5_[EI_t] = C5_interpolated
    return Dict_interpolated_V3_, Dict_interpolated_V5_, Dict_interpolated_C3_, Dict_interpolated_C5_

In [14]:
def compress_interp_35(Dict_interpolated_V3_tp1, Dict_interpolated_C3_tp1, Dict_interpolated_V5_tp1, Dict_interpolated_C5_tp1):
    Dict_interp = {}
    Dict_interp['V'] = {}
    Dict_interp['V'][3] = Dict_interpolated_V3_tp1
    Dict_interp['V'][5] = Dict_interpolated_V5_tp1
    Dict_interp['C'] = {}
    Dict_interp['C'][3] = Dict_interpolated_C3_tp1
    Dict_interp['C'][5] = Dict_interpolated_C5_tp1
    return Dict_interp

In [15]:
def calculate_Dicts_35(alpha_):
    numpoints = 30
    Dict_grids = {}
    for Hstate in [3, 5]:
        Dict_grids[Hstate] = gen_grid_35(alpha_, numpoints, Hstate)
    Dict_interpolated_V3 = {}
    Dict_interpolated_V5 = {}
    Dict_interpolated_C3 = {}
    Dict_interpolated_C5 = {}
    V3_= {}
    V5_= {}
    for t in range(40, 0, -1):
        EI_tm1_range = range(t)
        V3_current = {}
        V5_current = {}
        if t == 40:
            Dict_interpolated_V3_tp1 = {}
            Dict_interpolated_C3_tp1 = {}
            Dict_interpolated_V5_tp1 = {}
            Dict_interpolated_C5_tp1 = {}
        if t < 40:
            Dict_interpolated_V3_tp1 = Dict_interpolated_V3[t+1]
            Dict_interpolated_C3_tp1 = Dict_interpolated_C3[t+1]
            Dict_interpolated_V5_tp1 = Dict_interpolated_V5[t+1]
            Dict_interpolated_C5_tp1 = Dict_interpolated_C5[t+1]
        Dict_interp_tp1_35 = compress_interp_35(Dict_interpolated_V3_tp1, Dict_interpolated_C3_tp1, Dict_interpolated_V5_tp1, Dict_interpolated_C5_tp1)
        for EI_tm1 in EI_tm1_range:
            for W_t in Dict_grids[3][t]:
                key = (EI_tm1, W_t)
                V3_current[key] = V_t35(EI_tm1, W_t, alpha_, 3, t, Dict_interp_tp1_35)
            for W_t in Dict_grids[5][t]:
                key = (EI_tm1, W_t)
                V5_current[key] = V_t35(EI_tm1, W_t, alpha_, 5, t, Dict_interp_tp1_35)
        V3_[t] = V3_current
        V5_[t] = V5_current
        results = add_interpolated_CV_35(t, alpha_, Dict_grids, V3_current, V5_current)
        Dict_interpolated_V3[t] = results[0]
        Dict_interpolated_V5[t] = results[1]
        Dict_interpolated_C3[t] = results[2]
        Dict_interpolated_C5[t] = results[3]
        # print(t)
    return V3_, V5_, Dict_interpolated_V3, Dict_interpolated_V5, Dict_interpolated_C3, Dict_interpolated_C5

In [16]:
results = Parallel(n_jobs=25)(delayed(calculate_Dicts_35)(alphas[i]) for i in range(25))
Dict_35 = {}
for i in range(25):
    Dict_35[i] = {}
    Dict_35[i]['V3']  = [res[0] for res in results][i]
    Dict_35[i]['V5']  = [res[1] for res in results][i]
    Dict_35[i]['WV3'] = [res[2] for res in results][i]
    Dict_35[i]['WV5'] = [res[3] for res in results][i]
    Dict_35[i]['WC3'] = [res[4] for res in results][i]
    Dict_35[i]['WC5'] = [res[5] for res in results][i]
with open('Dict_35', 'wb') as file:
    pickle.dump(Dict_35, file)